# Tutorial 1 — Building a POPC Bilayer with Shapespyer

## Learning objectives

By the end of this tutorial, you will be able to:

1. Inspect the molecular template for a POPC lipid.
2. Generate a lipid bilayer using Shapespyer.
3. Visualise the generated membrane in three dimensions.
4. Explore how the number of lipids affects the membrane structure.

**Requirements:** Shapespyer, Jupyter, NumPy, nglview and ipywidgets.

**Working directory:** Run this notebook from `examples/workshops/`.

## Environment setup

This tutorial can run locally or in Google Colab.

**Google Colab:** Run the setup cell below to install Shapespyer
and download the POPC molecular template.

**Local Jupyter:** Use your existing Shapespyer environment and
run the notebook from `examples/workshops/`.

In [ ]:
import sys
import subprocess
from pathlib import Path
from urllib.request import urlretrieve

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    print("Google Colab detected. Installing Shapespyer...")

    subprocess.check_call([
        sys.executable, "-m", "pip", "install",
        "shapespyer[protein] @ git+https://gitlab.com/simnavi/shapespyer.git@workshop-tutorials"
    ])

    # Enable Colab's custom widget manager for NGLView.
    from google.colab import output
    output.enable_custom_widget_manager()

    Path("inputs").mkdir(exist_ok=True)

    popc_url = (
        "https://raw.githubusercontent.com/CCPBioSim/2026-building-molecular-systems-workshop/shapespyer-workshop/Shapespyer/"
        "inputs/POPC.pdb"
    )

    urlretrieve(popc_url, "inputs/POPC.pdb")

    protein_url = (
        "https://raw.githubusercontent.com/CCPBioSim/2026-building-molecular-systems-workshop/shapespyer-workshop/Shapespyer/"
        "inputs/"
        "bacteriorhodopsin_protein.pdb"
    )
    urlretrieve(protein_url, "inputs/bacteriorhodopsin_protein.pdb")

    # Download the small NAMD reference datasets for Tutorial 3.
    reference_dir = Path("outputs/tutorial_03/reference")
    reference_dir.mkdir(parents=True, exist_ok=True)

    reference_base = (
        "https://raw.githubusercontent.com/CCPBioSim/2026-building-molecular-systems-workshop/shapespyer-workshop/Shapespyer/"
        "outputs/tutorial_03/reference/"
    )

    for filename in (
        "bacteriorhodopsin_POPC_solvated_equil_100ps.xst",
        "bacteriorhodopsin_POPC_solvated_equil2_100ps.xst",
        "bacteriorhodopsin_POPC_solvated_prod_100ps.xst",
        "bacteriorhodopsin_POPC_prod_100ps_rmsd.dat",
    ):
        urlretrieve(reference_base + filename, reference_dir / filename)
        print("Downloaded reference:", filename)


else:
    print("Local Jupyter environment detected.")

assert Path("inputs/POPC.pdb").is_file(), (
    "POPC.pdb not found. Check the workshop input directory."
)

print("Workshop environment ready.")


## 1. Introduction

POPC (1-palmitoyl-2-oleoyl-sn-glycero-3-phosphocholine) is a
zwitterionic phospholipid commonly used to model mammalian cell membranes.

A lipid bilayer consists of two leaflets. The hydrophilic headgroups
face the surrounding aqueous environment, while the hydrophobic
tails point towards the membrane interior.

Shapespyer can generate the initial coordinates of a bilayer by
arranging copies of a molecular template into the requested geometry.

In this tutorial, we will use the supplied `POPC.pdb` structure.

## 2. Inspect the input molecule

The file `inputs/POPC.pdb` contains the coordinates of a POPC molecule.

We will first check that the file exists and inspect its contents.

In [ ]:
from pathlib import Path
import subprocess

import nglview as nv

input_file = Path("inputs/POPC.pdb")
output_dir = Path("outputs/tutorial_01")
output_dir.mkdir(parents=True, exist_ok=True)

assert input_file.is_file(), f"Missing input file: {input_file}"

print("Input:", input_file)
print("File size:", input_file.stat().st_size, "bytes")
print("\nFirst ten lines:")

print("\n".join(input_file.read_text().splitlines()[:10]))

### Visualise the POPC molecule

The interactive viewer allows you to rotate and zoom in on the
molecular structure.

In [ ]:
view = nv.show_structure_file(str(input_file))
view

## 3. Generate a POPC bilayer

We will now use the Shapespyer command-line interface.

The main options are:

- `--dio`: input and output directories.
- `-i`: input molecular template.
- `-o`: output file basename.
- `-x`: output format.
- `-s`: requested geometry.
- `-r`: lipid residue name.
- `--nside`: controls the number of molecules along each side.
- `--origin`: method used to position the molecular template.

The command below uses `--nside 10`, following the original
Shapespyer POPC example.

In [ ]:
!shape --dio=inputs,outputs/tutorial_01 -i POPC.pdb -o config -x .pdb -s bilayer -r POPC --nside 10 --origin cog

## 4. Inspect the generated bilayer

Shapespyer writes the molecular coordinates into a PDB file.

Let's locate the output and inspect the structure.

In [ ]:
generated_files = sorted(output_dir.glob("config*bilayer*.pdb"))

if not generated_files:
    raise FileNotFoundError(
        f"No generated bilayer PDB found in {output_dir}"
    )

bilayer_file = generated_files[0]
print("Generated structure:", bilayer_file)

view = nv.show_structure_file(str(bilayer_file))
view

## 5. Check the number of lipids

We can count the POPC residues in the generated PDB structure.

This gives us a simple way to check the output before moving
on to simulation preparation.

In [ ]:
residues = set()

with bilayer_file.open() as handle:
    for line in handle:
        if line.startswith(("ATOM  ", "HETATM")):
            if line[17:21].strip() == "POPC":
                residues.add(
                    (line[21:22], line[22:26].strip(), line[26:27])
                )

print("Number of POPC residues:", len(residues))

## 6. Hands-on exercise

Now modify the bilayer size.

**Exercise A:** Change `--nside 10` to `--nside 6`, and change
the output basename from `config` to `config_small`.

**Exercise B:** Generate a larger bilayer using `--nside 12`
and the output basename `config_large`.

For each structure:

1. Visualise the bilayer.
2. Count the POPC molecules.
3. Compare the approximate lateral dimensions.
4. Discuss how changing the bilayer size might affect
   the computational cost of an MD simulation.

**Question:** Does changing `--nside` affect only the number
of lipids, or also the dimensions of the generated membrane?

In [ ]:
# Exercise A — Generate a smaller POPC bilayer
# 6 × 6 lipids per leaflet = 72 POPC molecules

!shape --dio=inputs,outputs/tutorial_01 -i POPC.pdb -o config_small -x .pdb -s bilayer -r POPC --nside 6 --origin cog

## 7. Summary

You have now:

- Inspected a POPC molecular template.
- Generated a lipid bilayer with Shapespyer.
- Visualised the resulting structure.
- Explored how bilayer size can be controlled.

**Next tutorial:** Preparing the bilayer for NAMD simulation,
including PSF generation, solvation, ion addition and
simulation input files.

# Tutorial 2 — Preparing and Simulating a POPC Bilayer with NAMD

## Learning objectives

By the end of this tutorial, you will be able to:

1. Generate a NAMD-compatible PSF topology for a POPC bilayer.
2. Solvate the membrane and add ions at a specified salt concentration.
3. Prepare and run minimisation, equilibration and production simulations with NAMD.
4. Analyse the resulting trajectory using `ana trajectory`, including number-density and scattering-length-density profiles.

**Requirements:** Shapespyer, Jupyter, NumPy, nglview and ipywidgets.

**Working directory:** Run this notebook from `examples/workshops/`.

## 1. Introduction

In Tutorial 1, we constructed a POPC lipid bilayer containing 200 lipids
using Shapespyer. We will now prepare that structure for molecular
dynamics simulations with NAMD.

The structure generated in Tutorial 1 contains the atomic coordinates
of the membrane, but further preparation is required before simulation.

We will:

1. Generate a PSF (Protein Structure File) containing the molecular
   topology, including atoms, bonds, angles and dihedrals.
2. Add water molecules around the membrane and introduce NaCl ions.
3. Generate the topology of the complete solvated system.
4. Prepare NAMD configuration files for energy minimisation,
   equilibration and production.
5. Run a short demonstration simulation and examine its trajectory.

The PDB file describes the atomic coordinates, while the PSF provides
the molecular connectivity and topology required by NAMD. Force-field
parameter files supply the interaction parameters.

**Prerequisite:** Complete Tutorial 1 first. This notebook expects
`outputs/tutorial_01/config_POPC_bilayer_ns10_zs0A.pdb`.

**Important:** The short simulations used in this workshop are for
demonstration only and are not sufficient for production-quality
molecular dynamics.

## 2. Generate the initial PSF topology

We will use the POPC bilayer generated in Tutorial 1.

The first step is to create a NAMD-compatible PSF file containing
the molecular topology of the membrane.

In [ ]:
from pathlib import Path
import shutil
import nglview as nv
import numpy as np
import matplotlib.pyplot as plt

# Reuse the POPC bilayer generated in Tutorial 01.
input_pdb = Path("outputs/tutorial_01/config_POPC_bilayer_ns10_zs0A.pdb")

assert input_pdb.is_file(), (
    "The POPC bilayer from Tutorial 01 was not found. "
    "Please complete Tutorial 01 in the same Colab runtime "
    "before starting Tutorial 02."
)

output_dir = Path("outputs/tutorial_02")
output_dir.mkdir(parents=True, exist_ok=True)

bilayer_pdb = output_dir / input_pdb.name
shutil.copy2(input_pdb, bilayer_pdb)

print("Using the bilayer generated in Tutorial 01:")
print(bilayer_pdb)

### Generate the PSF

The `namd-setup-psf` command creates a topology file from the
bilayer coordinates.

Both the PDB and PSF files will be used during subsequent
solvation and simulation steps.

In [ ]:
!namd-setup-psf --inp outputs/tutorial_02/config_POPC_bilayer_ns10_zs0A.pdb --out outputs/tutorial_02/config_POPC_bilayer_ns10_zs0A.psf

## 3. Solvate and ionise the POPC bilayer

A lipid membrane must be surrounded by water before performing
molecular dynamics simulations.

We use `namd-solvate` to:

- Add water layers extending 20 Å along the membrane normal.
- Introduce NaCl at a nominal concentration of 0.15 M (150 mM).
- Produce a PDB containing the membrane, water and ions.

**Note:** The `--saltconc` option expects mol/L in the current
implementation, despite the incorrect units shown in its help text.

In [ ]:
!namd-solvate --inp outputs/tutorial_02/config_POPC_bilayer_ns10_zs0A.pdb,outputs/tutorial_02/config_POPC_bilayer_ns10_zs0A.psf --out outputs/tutorial_02/POPC-TIP3P-NA-CL_bilayer.pdb --zlayer 20 --saltconc 0.15 --layer_type bilayer

### Visualise the solvated membrane

Inspect the resulting structure. Notice how the water molecules
surround both sides of the POPC bilayer.

In [ ]:
solvated_pdb = output_dir / "POPC-TIP3P-NA-CL_bilayer.pdb"

assert solvated_pdb.is_file(), "Solvated PDB not found."

view = nv.show_structure_file(str(solvated_pdb))
view.clear_representations()

# POPC bilayer
view.add_licorice(
    selection="POPC",
    color="element",
    radius_scale=0.6
)

# TIP3 water oxygen atoms
view.add_spacefill(
    selection="TIP3 and _O",
    color="dodgerblue",
    radius_scale=0.3,
    opacity=0.7
)

# Sodium and chloride ions
view.add_spacefill(
    selection="SOD or CLA",
    color="element",
    radius_scale=0.8
)

view.center()
view

## 4. Generate the topology of the solvated system

After adding water and ions, the original PSF is no longer sufficient:
it describes only the POPC bilayer.

We must generate a new PSF containing all atoms in the complete
system, including the membrane, water molecules and ions.

The resulting PDB and PSF files will be used to prepare the NAMD
simulation.

In [ ]:
!namd-setup-psf --inp outputs/tutorial_02/POPC-TIP3P-NA-CL_bilayer.pdb --out outputs/tutorial_02/POPC-TIP3P-NA-CL_bilayer.psf

### Verify the generated topology

Check that both the PDB and PSF files exist before proceeding.

In [ ]:
final_pdb = output_dir / "POPC-TIP3P-NA-CL_bilayer.pdb"
final_psf = output_dir / "POPC-TIP3P-NA-CL_bilayer.psf"

assert final_pdb.is_file(), "Final PDB file not found."
assert final_psf.is_file(), "Final PSF file not found."

print(f"PDB: {final_pdb}")
print(f"PSF: {final_psf}")
print("The solvated system is ready for NAMD configuration.")

## 5. Prepare the NAMD simulation

Shapespyer can generate the configuration files required for three
successive simulation stages:

1. **Energy minimisation** — relaxes the initial structure.
2. **Equilibration** — allows the solvated membrane to adjust.
3. **Production** — generates the trajectory used for analysis.

We will prepare short example configurations to demonstrate the workflow.

**NAMD is not required for this workshop.** We will use precomputed
simulation results in the following sections.

In [ ]:
from pathlib import Path
from urllib.request import urlretrieve

# Download the CHARMM parameter files required by NAMD.
toppar_dir = Path("toppar")
toppar_dir.mkdir(exist_ok=True)

parameter_base_url = (
    "https://raw.githubusercontent.com/CCPBioSim/2026-building-molecular-systems-workshop/shapespyer-workshop/Shapespyer/"
    "toppar/"
)

parameter_files = (
    "par_all36_lipid.prm",
    "par_all36_prot.prm",
    "par_all27_prot_na.prm",
)

for filename in parameter_files:
    destination = toppar_dir / filename

    if not destination.is_file():
        urlretrieve(parameter_base_url + filename, destination)

    assert destination.is_file() and destination.stat().st_size > 0, (
        f"Missing or empty CHARMM parameter file: {destination}"
    )
    print(f"Ready: {destination}")

# Download NAMD configuration templates for Colab.
template_base_url = (
    "https://raw.githubusercontent.com/CCPBioSim/2026-building-molecular-systems-workshop/shapespyer-workshop/Shapespyer/"
    ""
)

template_files = (
    "min_template.conf",
    "equil_template.conf",
    "prod_template.conf",
)

for filename in template_files:
    destination = Path(filename)

    if not destination.is_file():
        urlretrieve(template_base_url + filename, destination)

    assert destination.is_file() and destination.stat().st_size > 0, (
        f"Missing or empty NAMD template: {destination}"
    )
    print(f"Ready: {destination}")

### Generate minimisation, equilibration and production configurations

For this demonstration, we request 100 minimisation steps and
1,000 steps for each MD stage. These short runs are intended only
to test the workflow, not to produce scientifically equilibrated results.

In [ ]:
!namd-setup-equil --struct outputs/tutorial_02/POPC-TIP3P-NA-CL_bilayer --out_min outputs/tutorial_02/POPC_min.conf --out_equil outputs/tutorial_02/POPC_equil.conf --out_prod outputs/tutorial_02/POPC_prod.conf --temp 300 --steps_min 100 --steps_equil 1000 --steps_prod 1000

In [ ]:
for filename in ("POPC_min.conf", "POPC_equil.conf", "POPC_prod.conf"):
    config = output_dir / filename
    assert config.is_file(), f"Missing configuration: {config}"
    print(f"Generated: {config}")

### Prepare the short demonstration minimisation

We will run **100 minimisation steps** to demonstrate how NAMD works.

This is not sufficient to relax the membrane fully. Longer minimisation,
equilibration and production results will be provided separately.

We use a different output prefix (`POPC_demo_min`) so the demonstration
cannot overwrite the reference simulation results.

In [ ]:
from pathlib import Path

original_config = output_dir / "POPC_min.conf"
demo_config = output_dir / "POPC_demo_min.conf"

content = original_config.read_text()

# NAMD needs paths that resolve correctly in the Colab environment.
content = content.replace(
    "outputs/tutorial_02/",
    str(output_dir.resolve()) + "/"
)
content = content.replace(
    "toppar/",
    str(Path("toppar").resolve()) + "/"
)

# Give the demonstration its own output files.
lines = content.splitlines()
lines = [
    f"outputName {output_dir.resolve() / 'POPC_demo_min'}"
    if line.lstrip().startswith("outputName")
    else line
    for line in lines
]

demo_config.write_text("\n".join(lines) + "\n")

print(f"Demonstration configuration: {demo_config}")
print(f"Output prefix: {output_dir.resolve() / 'POPC_demo_min'}")

## 6. Optional: Run a short NAMD energy minimisation

This optional exercise demonstrates 100 steps of energy minimisation
using NAMD 3.

**The default workshop does not require NAMD.** You can run all cells
and continue to Sections 7 and 8 using the precomputed trajectory.

100 minimisation steps are for demonstration only and are not
sufficient for production-quality molecular dynamics.

### Download and install your own NAMD copy

To run the optional minimisation in Google Colab:

1. Visit the [official NAMD website](https://www.ks.uiuc.edu/Research/namd/).
2. Register and follow the official download instructions.
3. Download **NAMD 3.0.2, Linux x86-64 multicore**.
4. Save `NAMD_3.0.2_Linux-x86_64-multicore.tar.gz`
   on your computer.
5. In Colab, open the **Files** panel on the left.
6. Right-click the `/content` folder and select **Upload**.
7. Upload the downloaded `.tar.gz` archive.
8. Change `RUN_NAMD = False` to `RUN_NAMD = True` below.
9. Run the installation cell and then the minimisation cell.

The installation cell extracts the archive into `/content`
and locates the `namd3` executable automatically.

For simulations on your own computer, obtain the appropriate
NAMD distribution for your operating system and architecture.

NAMD is distributed separately from Shapespyer.

In [ ]:
from pathlib import Path
import tarfile

# Enable only if you have downloaded and uploaded NAMD.
RUN_NAMD = False

namd_exe = None

if RUN_NAMD:
    archive = Path(
        "/content/NAMD_3.0.2_Linux-x86_64-multicore.tar.gz"
    )
    namd_exe = Path(
        "/content/NAMD_3.0.2_Linux-x86_64-multicore/namd3"
    )

    if not namd_exe.is_file():
        assert archive.is_file(), (
            f"Missing NAMD archive: {archive}. "
            "Download it from the official website and "
            "upload it to /content."
        )

        print("Extracting NAMD...")
        with tarfile.open(archive, "r:gz") as tar:
            tar.extractall(path="/content", filter="data")

    assert namd_exe.is_file(), (
        f"NAMD executable not found: {namd_exe}"
    )
    print(f"NAMD executable: {namd_exe}")
else:
    print("Optional NAMD installation skipped.")
    print("You can continue directly to Section 7.")

### Optional: Run the 100-step minimisation

If `RUN_NAMD = True`, the following cell runs the demonstration
using `POPC_demo_min.conf`, prepared in Section 5.

The simulation takes approximately two minutes on a typical
Colab CPU runtime, although execution time may vary.

Its output prefix is `POPC_demo_min`, so it will not overwrite
the precomputed reference simulation.

If `RUN_NAMD = False`, the cell skips the simulation automatically.

In [ ]:
import subprocess

if RUN_NAMD:
    assert namd_exe is not None and namd_exe.is_file(), (
        "NAMD is not installed. Run the preceding installation cell."
    )

    config = output_dir / "POPC_demo_min.conf"
    log_file = output_dir / "POPC_demo_min.log"

    with log_file.open("w") as log:
        result = subprocess.run(
            [str(namd_exe), "+p2", str(config)],
            stdout=log,
            stderr=subprocess.STDOUT,
            text=True,
            timeout=600,
        )

    print("NAMD return code:", result.returncode)
    print("\nLast 20 lines of the log:")
    print("\n".join(
        log_file.read_text(errors="replace").splitlines()[-20:]
    ))

    assert result.returncode == 0, (
        f"NAMD minimisation failed. Inspect {log_file}"
    )
else:
    print("NAMD minimisation skipped (optional exercise).")
    print("Continue to Section 7 for trajectory analysis.")

### What did we obtain?

NAMD writes several files containing the minimised coordinates,
velocities and simulation-cell information.

These files can be used to restart a simulation, but the structure
is **not yet sufficiently minimised for production dynamics**.

In a complete simulation workflow, we would continue with longer
minimisation, equilibration and production runs.

For the remainder of this workshop, we will use a precomputed
POPC trajectory so that everyone can explore trajectory analysis
without waiting for a full molecular dynamics simulation.

## 7. Visualise a precomputed POPC trajectory

Running a molecular dynamics simulation can take considerable time.
To make this workshop accessible without NAMD, we will use a
precomputed POPC simulation included in the Shapespyer examples.

The reference dataset contains:

- A PDB structure describing the simulated system.
- A PSF topology containing the molecular connectivity.
- A production DCD trajectory containing 100 saved frames.

**Note:** This reference simulation is separate from the system
generated earlier in this tutorial. We therefore use its own matching
structure and trajectory files.

### Load the reference trajectory

We use MDTraj to read the trajectory and NGLView to visualise it.

In [ ]:
from pathlib import Path
from urllib.request import urlretrieve

reference_dir = Path("outputs/tutorial_02/reference")
reference_dir.mkdir(parents=True, exist_ok=True)

reference_base_url = (
    "https://raw.githubusercontent.com/CCPBioSim/2026-building-molecular-systems-workshop/shapespyer-workshop/Shapespyer/"
    "reference/"
)

for filename in (
    "POPC_reference.pdb",
    "POPC_reference.psf",
    "POPC_reference_prod.dcd",
):
    destination = reference_dir / filename

    if not destination.is_file():
        print(f"Downloading {filename}...")
        urlretrieve(reference_base_url + filename, destination)

    assert destination.is_file() and destination.stat().st_size > 0, (
        f"Missing or empty reference file: {destination}"
    )

    print(f"Ready: {destination}")

In [ ]:
import mdtraj as md

reference_pdb = reference_dir / "POPC_reference.pdb"
reference_psf = reference_dir / "POPC_reference.psf"
reference_dcd = reference_dir / "POPC_reference_prod.dcd"

for path in (reference_pdb, reference_psf, reference_dcd):
    assert path.is_file(), f"Missing reference file: {path}"

trajectory = md.load(str(reference_dcd), top=str(reference_pdb))

print(f"Atoms: {trajectory.n_atoms}")
print(f"Residues: {trajectory.n_residues}")
print(f"Frames: {trajectory.n_frames}")

### Explore membrane dynamics

The interactive viewer below displays the POPC bilayer from the
production trajectory.

Use the playback controls to move between saved frames.
You can rotate and zoom the structure using your mouse.

For clarity, water molecules and ions are hidden in this initial view.

In [ ]:
# Select POPC atoms using MDTraj
import ipywidgets as widgets
from IPython.display import display

lipid_atoms = trajectory.topology.select("resname POPC")
lipid_trajectory = trajectory.atom_slice(lipid_atoms)

print(f"Selected {lipid_trajectory.n_atoms} atoms from 200 POPC molecules.")

# Display only the lipid bilayer
trajectory_view = nv.show_mdtraj(lipid_trajectory)
trajectory_view.clear_representations()
trajectory_view.add_ball_and_stick()
trajectory_view.center()

# Custom trajectory slider for reliable playback in Google Colab
frame_slider = widgets.IntSlider(
    value=0,
    min=0,
    max=lipid_trajectory.n_frames - 1,
    step=1,
    description="Frame:",
    continuous_update=True
)

def update_frame(change):
    if change["name"] == "value":
        trajectory_view.frame = change["new"]

frame_slider.observe(update_frame, names="value")

display(trajectory_view)
display(frame_slider)

### Things to observe

As you play the trajectory, consider:

- Do the POPC molecules remain organised into two leaflets?
- How do the lipid tails fluctuate over time?
- Does the membrane remain approximately planar?
- Why is it important to equilibrate a membrane before analysing
  its structural properties?

The next section will introduce quantitative trajectory analysis
using Shapespyer's `ana trajectory` functionality.

## 8. Density Profiles: Connecting Molecular Dynamics to Neutron Reflectometry

A molecular dynamics trajectory provides atomic coordinates at successive
simulation times. We can use these coordinates to calculate how the
different components of the membrane are distributed along the membrane
normal (the z-axis).

In this section, we will use Shapespyer's `ana-trajectory` command to
calculate:

- **Number density (ND):** the number of atoms or molecular groups per unit volume.
- **Neutron scattering-length density (SLD):** the coherent neutron
  scattering length per unit volume.

We will examine the distributions of lipid atoms, water and the total
neutron SLD profile.

For this workshop, we use a compact reference trajectory containing 20 frames sampled from a longer production simulation. We analyse all 20 frames to demonstrate the calculation of density profiles. These results are illustrative and should not be interpreted as statistically converged.

In [ ]:
from pathlib import Path

analysis_dir = Path("outputs/tutorial_02")
analysis_dir.mkdir(parents=True, exist_ok=True)

# Use the precomputed reference trajectory downloaded in Section 7.
reference_pdb = reference_dir / "POPC_reference.pdb"
reference_dcd = reference_dir / "POPC_reference_prod.dcd"

assert reference_pdb.is_file(), f"Missing reference PDB: {reference_pdb}"
assert reference_dcd.is_file(), f"Missing reference DCD: {reference_dcd}"

!ana-trajectory \
    --inp {reference_dcd} \
    --ref {reference_pdb} \
    --rnames POPC \
    --wname TIP3 \
    --dtypes nden,nsld \
    --zbin 0.1 \
    --freq 1

print("Density analysis completed.")

### Understanding the output files

Shapespyer saves the calculated profiles as two-column `.dat` files:
the z-position (nm) and the corresponding density value.

For example, `POPC_reference_prod_znden_ATOM_P_POPC_avr.dat`
contains the average number-density profile of phosphorus atoms in POPC.

The filenames distinguish the density type (`znden`, `znsld` or `zmden`),
the representation (`ATOM` or `UAGR`), the chemical component and,
where applicable, the residue (`POPC` or `TIP3`).

The neutron SLD is expressed in units of 10⁻⁶ Å⁻².

### Plotting the density profiles

We will first examine the distributions of phosphorus, carbon and water
oxygen atoms across the bilayer. These profiles help us identify the
lipid headgroup regions, hydrocarbon interior and surrounding water.

Next, we will plot the total neutron SLD calculated from the atomic
and united-atom representations.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

profile_dir = Path("outputs/tutorial_02/reference")
prefix = "POPC_reference_prod"

def load_profile(suffix):
    filename = profile_dir / f"{prefix}_{suffix}_avr.dat"
    return np.loadtxt(filename, unpack=True)

fig, ax = plt.subplots(figsize=(9, 5))

for suffix, label in [
    ("znden_ATOM_P_POPC", "POPC phosphorus"),
    ("znden_ATOM_C_POPC", "POPC carbon"),
    ("znden_ATOM_O_TIP3", "Water oxygen"),
]:
    z, density = load_profile(suffix)
    ax.plot(z, density, label=label)

ax.set_xlabel("Position along membrane normal, z (nm)")
ax.set_ylabel("Number density (atoms/nm³)")
ax.set_title("Atomic number-density profiles across the POPC bilayer")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

for suffix, label in [
    ("znsld_ATOMS", "Atomic representation"),
    ("znsld_UAGRS", "United-atom representation"),
]:
    z, sld = load_profile(suffix)
    ax.plot(z, sld, label=label)

ax.axhline(0, color="gray", linewidth=0.8)
ax.set_xlabel("Position along membrane normal, z (nm)")
ax.set_ylabel("Neutron SLD (10⁻⁶ Å⁻²)")
ax.set_title("Neutron scattering-length density of the POPC bilayer")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### Interpreting the results

Consider the following questions:

1. Why does phosphorus have two peaks on opposite sides of the bilayer?
2. Where is the carbon density highest, and what does this tell us
   about the lipid tails?
3. How does the water distribution change towards the bilayer centre?
4. Why can the neutron SLD be negative in hydrogen-rich regions?
5. How closely do the atomic and united-atom SLD profiles agree?

**Note:** These profiles use a small subset of the reference trajectory
for demonstration. More frames are needed for statistically reliable
density profiles.

# Tutorial 3 — Building and Preparing a Membrane–Protein System with Shapespyer

## Learning objectives

By the end of this tutorial, you will be able to:

1. Insert a membrane protein into a POPC lipid bilayer using Shapespyer.
2. Identify and reconstruct missing protein residues before simulation.
3. Prepare a protein–lipid system and generate a CHARMM-compatible PSF topology.
4. Solvate and ionise the combined system.
5. Understand the protein-aware minimisation and equilibration workflow in NAMD.

**Requirements:** Shapespyer, Jupyter, and the molecular visualisation tools used in Tutorials 1 and 2. Protein reconstruction requires PDBFixer through the established preparation workflow.

**Working directory:** Run this notebook from `examples/workshops/`.

**Status:** Workshop draft with executable preparation, validation and NAMD configuration cells; the complete notebook has not yet been validated end-to-end.

## 1. Introduction

In Tutorial 1, we built a POPC lipid bilayer using Shapespyer. In Tutorial 2, we prepared a lipid-only membrane for molecular dynamics (MD), including PSF generation, solvation, ion addition, and NAMD configuration.

Real biological membranes also contain **membrane proteins**, which carry out functions such as transport, signalling, and energy conversion. Simulating a membrane protein requires us to prepare **both** the protein and its surrounding lipid environment. This adds several challenges beyond a lipid-only system: placing the protein correctly, checking its sequence and missing atoms, preventing unfavourable protein–lipid contacts, and choosing a staged equilibration protocol.

In this tutorial, our example is **apo-bacteriorhodopsin embedded in a POPC bilayer**. Bacteriorhodopsin is a seven-transmembrane-helix protein that, in its native retinal-bound form, uses light energy to pump protons across a membrane. **Our demonstration does not include retinal**: it is a structural preparation and MD-workflow exercise, not a simulation of light-driven proton pumping.

We will build on the same Shapespyer and NAMD tools introduced earlier, rather than constructing a separate workflow from scratch.

## 2. Overview of the workflow

We will follow these stages:

1. **Insert the protein.** Place bacteriorhodopsin within a POPC bilayer using the Shapespyer `shape` command.
2. **Check and repair the protein.** Compare the structural sequence with the reference sequence, reconstruct missing residues, and prepare the protein for the force field.
3. **Prepare the combined structure.** Generate the protein–lipid PDB and its PSF topology, checking atom order and peptide connectivity.
4. **Solvate and add ions.** Add water on both sides of the membrane and introduce NaCl at a nominal concentration of 0.15 M.
5. **Generate protein-aware NAMD configurations.** Use `namd-setup-equil --protein-equil` for staged minimisation, two equilibration stages, and production.
6. **Inspect the results.** Visualise the prepared structure and use precomputed reference results from a short validated test run to illustrate basic simulation checks.

**Why two equilibration stages?** The initial protocol restrains the lipid phosphorus atoms and protein backbone while the surroundings relax. A second equilibration stage releases the protein backbone restraints while retaining the lipid phosphorus restraints. Production is unrestrained. These stages help avoid introducing all degrees of freedom simultaneously; they do not guarantee a physically equilibrated system.

## 3. Our example system

The prepared test system contains:

| Component | Quantity |
|---|---:|
| POPC lipids | 200 |
| Protein residues | 227 |
| TIP3 water molecules | 11,337 |
| Sodium ions (SOD) | 31 |
| Chloride ions (CLA) | 31 |
| Total atoms | 64,448 |

The protein was prepared from PDB **1C3W** with reference to UniProt **P02945**. Five missing residues (**TSKAE**, positions 157–161) were reconstructed before preparing the solvated system. The resulting PSF was checked for all 226 consecutive peptide bonds.

These values describe the **current prepared example**, not a guarantee that every fresh build will produce identical coordinates or solvent counts. The missing loop still requires physical assessment after minimisation and equilibration.

## 4. Before we begin

In the next sections we will introduce the relevant Shapespyer commands, explain their inputs and outputs, and inspect the resulting structures. As in Tutorial 2, workshop participants should not need to wait for a long MD simulation: a short live demonstration can be paired with separately supplied, matching reference results.

**Validation note (9 November 2026):** The reference system successfully completed 20,000 minimisation steps, 111 ps of first-stage equilibration, 100 ps of second-stage equilibration and 100 ps of unrestrained NPT production using workshop-specific CHARMM36 water/ion parameters. The complete notebook still requires end-to-end testing; successful simulation execution does not establish physical equilibrium.

**Next:** Section 5 introduces protein insertion; subsequent sections will cover protein repair and topology preparation.

## 5. Insert bacteriorhodopsin into a POPC bilayer

We will now use the `shape` command introduced in Tutorial 1, but supply a **protein structure** in addition to the POPC molecular template. Shapespyer arranges the lipids into a bilayer and creates space for the protein according to its footprint.

Our input protein is based on **PDB 1C3W**. At this stage it is the *unrepaired* structure: we will identify and reconstruct its missing five-residue segment in the next section. This initial insertion is **not yet a simulation-ready model**.

For this demonstration we use `--nside 10`, `--zsep 3.0`, and the same protein-orientation and footprint settings used for the validated Tutorial 3 preparation. The exact meaning and suitability of these parameters should be checked for other proteins.

**Inputs:** `inputs/POPC.pdb` and `bacteriorhodopsin_protein.pdb`, both relative to  `examples/workshops/`. The protein input must be provided with the workshop materials; it is not generated by Tutorials 1 or 2.

**Output:** an initial protein–POPC PDB in `outputs/tutorial_03/`.

In [ ]:
from pathlib import Path

popc_input = Path('inputs/POPC.pdb')
protein_input = Path('inputs/bacteriorhodopsin_protein.pdb')
output_dir_t3 = Path('outputs/tutorial_03')
output_dir_t3.mkdir(parents=True, exist_ok=True)

assert popc_input.is_file(), f'Missing POPC template: {popc_input}'
assert protein_input.is_file(), (
    f'Missing protein input: {protein_input}. '
    'Add the workshop bacteriorhodopsin_protein.pdb file before proceeding.'
)
print('POPC template:', popc_input)
print('Protein structure:', protein_input)
print('Output directory:', output_dir_t3)

### Generate the protein-containing bilayer

The `--prot` option supplies the protein, while `--prot-footprint circle` defines the shape used to make room for it in the lipid assembly. `--alpha`, `--theta` and `--prot-zshift` control the initial protein placement and orientation. We use the parameters from the development run rather than assuming they are universally appropriate.

**Note:** Protein insertion removes lipids from the protein footprint; the resulting structure must be inspected for steric clashes and correct orientation. We will prepare the protein and repair the missing loop *after* this insertion step.

In [ ]:
!shape -v --dio=inputs,outputs/tutorial_03 -i POPC.pdb -o bacteriorhodopsin_POPC -x .pdb -r POPC -s bilayer --zsep 3.0 --nside 10 --prot inputs/bacteriorhodopsin_protein.pdb --alpha 0 --theta 90 --prot-zshift 0.0 --prot-lipid-ratio 0.005 --prot-footprint circle

### Locate and visualise the inserted system

We will use **NGLView**, as in Tutorials 1 and 2, to examine the protein in the membrane. Lipids are shown separately from the protein so that you can assess its orientation and placement.

Look for the seven transmembrane helices, the two lipid leaflets and any obvious protein–lipid overlap. Visual inspection is useful but does **not** replace quantitative contact checks.

In [ ]:
import nglview as nv

inserted_candidates = sorted(output_dir_t3.glob('bacteriorhodopsin_POPC*bilayer*.pdb'))
assert len(inserted_candidates) == 1, (
    f'Expected one inserted bilayer PDB, found {len(inserted_candidates)}: '
    f'{inserted_candidates}'
)
inserted_pdb = inserted_candidates[0]
print('Inserted structure:', inserted_pdb)

view = nv.show_structure_file(str(inserted_pdb))
view.clear_representations()
view.add_cartoon(selection='protein', color='spectrum')
view.add_licorice(selection='POPC', color='element', radius_scale=0.45)
view.center()
view

### Check the initial system

In the development run, this step produced a bilayer containing **200 POPC molecules** and an inserted bacteriorhodopsin structure. Your output should be inspected rather than assumed identical.

**Discussion:** Why is it important to insert the protein *before* reconstructing the missing loop? In our workflow, loop reconstruction was performed on the already-inserted protein so its geometry could be checked against the surrounding lipids.

**Next:** Extract the inserted protein, compare its sequence with UniProt P02945, and reconstruct residues 157–161 (`TSKAE`). These operations will be developed in Section 6; do not treat this initial structure as ready for NAMD.

## 6. Identify and reconstruct the missing protein loop

An experimentally determined protein structure may not contain every residue in the biological sequence. Missing residues are particularly important for membrane-protein simulations because an incomplete loop can leave an artificial break in the peptide backbone.

Our starting structure is based on **PDB 1C3W**. Comparing its deposited sequence with **UniProt P02945** revealed a missing five-residue segment, **TSKAE (residues 157–161)**, between PHE156 and SER162. This was not detected automatically by PDBFixer: the residues are absent from the deposited sequence information as well as the coordinates.

We will extract the protein **after insertion**, inspect its sequence, reconstruct the missing segment with PDBFixer, and examine the resulting loop in the membrane. This is an example-specific repair, **not** a generic algorithm for discovering sequence deletions.

### 6.1 Extract the inserted protein

Keep only protein coordinate records from the combined PDB, preserving their original coordinates. We use the protein residue names rather than relying on the PDB chain identifier (which may be absent or changed by the builder). The extraction also avoids copying the bilayer's `CRYST1` header as though it belonged to an isolated protein.

In [ ]:
from pathlib import Path

protein_resnames = {
    'ALA', 'ARG', 'ASN', 'ASP', 'CYS', 'GLN', 'GLU', 'GLY', 'HIS',
    'ILE', 'LEU', 'LYS', 'MET', 'PHE', 'PRO', 'SER', 'THR', 'TRP',
    'TYR', 'VAL', 'HSD', 'HSE', 'HSP'
}
inserted_protein = output_dir_t3 / 'bacteriorhodopsin_inserted.pdb'
protein_lines = [
    line for line in inserted_pdb.read_text().splitlines()
    if line.startswith(('ATOM  ', 'HETATM'))
        and len(line.split()) >= 5
    and line.split()[3] in protein_resnames
]
assert protein_lines, 'No protein atoms found: inspect the inserted PDB naming.'
inserted_protein.write_text('\n'.join(line[:17] + line[18:] for line in protein_lines) + '\nEND\n')
print('Extracted protein atoms:', len(protein_lines))
print('Saved:', inserted_protein)

### 6.2 Inspect the sequence gap

**Reference-sequence alignment is an essential prerequisite.** PDBFixer can fill missing residues reported by the structure, but it cannot infer a deletion that is also absent from the deposited sequence. For this particular 1C3W-based input, our prior comparison with UniProt P02945 identified the missing `TSKAE` segment.

The following cell lists the protein residues present in the extracted PDB. Check their numbering and sequence against the reference before attempting any repair; PDB residue identifiers and zero-based PDBFixer insertion indices are **not interchangeable**.

In [ ]:
three_to_one = {
    'ALA':'A','ARG':'R','ASN':'N','ASP':'D','CYS':'C','GLN':'Q',
    'GLU':'E','GLY':'G','HIS':'H','HSD':'H','HSE':'H','HSP':'H',
    'ILE':'I','LEU':'L','LYS':'K','MET':'M','PHE':'F','PRO':'P',
    'SER':'S','THR':'T','TRP':'W','TYR':'Y','VAL':'V'
}
residues = []
seen = set()
for line in protein_lines:
    key = (line[21:22], line[22:26].strip(), line[26:27], line[18:21].strip())
    if key not in seen:
        seen.add(key)
        residues.append(key)
print('Residues found:', len(residues))
print('Extracted sequence:')
print(''.join(three_to_one.get(r[3], 'X') for r in residues))
print('\nCheck the sequence against UniProt P02945; expected missing segment: TSKAE.')

### 6.3 Reconstruct the five missing residues with PDBFixer

In the development workflow, we loaded the **inserted protein** with PDBFixer and explicitly supplied `['THR', 'SER', 'LYS', 'ALA', 'GLU']` at insertion key `(0, 152)`. This key was determined for that **specific extracted structure**; it is **not** the biological residue number and must not be reused blindly on another PDB.

The cell below is guarded: it checks the chain count and the residues immediately around the proposed insertion index. If those checks fail, stop and inspect the structure and reference alignment rather than forcing a loop into the wrong location. PDBFixer builds an initial geometry; it does not establish that the loop is physically relaxed.

In [ ]:
from pdbfixer import PDBFixer
from openmm.app import PDBFile

fixer = PDBFixer(filename=str(inserted_protein))
chains = list(fixer.topology.chains())
assert len(chains) == 1, (
    f'Expected one protein chain; found {len(chains)}. '
    'Inspect the input before choosing a PDBFixer insertion key.'
)
chain_residues = list(chains[0].residues())
insertion_index = 152  # verified only for the original 1C3W-derived input
assert len(chain_residues) > insertion_index
before = chain_residues[insertion_index - 1]
after = chain_residues[insertion_index]
print('Residues adjacent to insertion:', before, after)
assert before.name == 'PHE' and after.name == 'SER', (
    'The expected PHE/SER junction was not found. '
    'Recheck the sequence alignment and insertion index.'
)

fixer.findMissingResidues()
assert not fixer.missingResidues, (
    'Additional missing residues detected; inspect them before proceeding.'
)
fixer.missingResidues[(0, insertion_index)] = [
    'THR', 'SER', 'LYS', 'ALA', 'GLU'
]
fixer.findMissingAtoms()
fixer.addMissingAtoms()

reconstructed_pdb = output_dir_t3 / 'bacteriorhodopsin_loop_reconstructed.pdb'
with reconstructed_pdb.open('w') as handle:
    PDBFile.writeFile(fixer.topology, fixer.positions, handle, keepIds=True)
print('Reconstructed protein:', reconstructed_pdb)
print('Residues after repair:', sum(1 for _ in fixer.topology.residues()))

### 6.4 Visualise and assess the reconstructed loop

Visualise the reconstructed protein and compare it with the initial structure. In our development checks, the inserted loop had **no heavy-atom contacts closer than 3 Å to POPC** (minimum separation approximately **5.38 Å**), and the resulting topology contained all **226 consecutive peptide bonds** after subsequent PSF generation. These results apply to the previously prepared example, **not automatically to this fresh run**.

**Discussion:** Why does a plausible-looking loop still require minimisation and structural checks? Think about peptide geometry, local strain, steric contacts and the surrounding lipid environment.

**Next:** use `namd-prepare-protein` to prepare the reconstructed protein, merge it back with the POPC coordinates while preserving atom order, and generate the combined PSF. We will cover those steps in Section 7.

In [ ]:
repaired_view = nv.show_structure_file(str(reconstructed_pdb))
repaired_view.clear_representations()
repaired_view.add_cartoon(color='spectrum')
repaired_view.add_licorice(selection='protein', radius_scale=0.4)
repaired_view.center()
repaired_view

## 7. Prepare the protein and build the protein–lipid topology

The reconstructed loop is only an initial structural model. Before NAMD can use the system, we must prepare the protein for the selected force field, combine it with the POPC coordinates, and generate a matching **PDB/PSF pair**. The PSF describes atom types, charges and molecular connectivity; the PDB supplies coordinates.

In this section we will:

1. Prepare the reconstructed bacteriorhodopsin using `namd-prepare-protein` at pH 7.0.
2. Merge the prepared protein and POPC coordinates, without accidentally duplicating the unrepaired protein.
3. Generate a CHARMM-compatible PSF using `namd-setup-psf`.
4. Check atom counts, atom ordering and peptide connectivity.

**Important:** We use the **reconstructed** protein from Section 6, not the original extracted protein. The following is a reproducible workshop draft; the precise commands and topology must still be validated end-to-end in the notebook environment.

### 7.1 Prepare the reconstructed protein

`namd-prepare-protein` applies the protein-preparation workflow (including protonation choices) to the repaired protein. The example uses **pH 7.0**. Review any warnings and protonation choices, particularly for a membrane protein with buried ionisable residues. For bacteriorhodopsin, this is a workflow demonstration, not a validated functional protonation model.

The command writes the prepared protein to `outputs/tutorial_03/`.

In [ ]:
prepared_protein = output_dir_t3 / 'bacteriorhodopsin_loop_prepared.pdb'
assert reconstructed_pdb.is_file(), f'Missing reconstructed protein: {reconstructed_pdb}'
print('Input:', reconstructed_pdb)
print('Prepared protein output:', prepared_protein)

In [ ]:
!namd-prepare-protein -i {reconstructed_pdb} -o {prepared_protein} --ph 7.0

### 7.2 Combine the prepared protein with the POPC bilayer

The initial Shapespyer output already contains **both** protein and lipids. We therefore retain only the **POPC** records from that structure and replace its original protein with the newly prepared protein.

The simple PDB merge below assumes the workshop input contains only one protein and POPC (no additional cofactors or other components). It preserves the atom records and coordinates but gives the protein and lipid residues distinct chain identifiers. **Do not use this approach for a system with additional molecular species without adapting the selection.**

We also check that the prepared protein is present and that the merged PDB contains exactly the expected number of POPC residues.

In [ ]:
from collections import Counter

assert prepared_protein.is_file(), f'Missing prepared protein: {prepared_protein}'
assert inserted_pdb.is_file(), f'Missing insertion output: {inserted_pdb}'

# Use only coordinate records; avoid carrying duplicate END/TER/CONECT records.
def pdb_atom_lines(path):
    return [line for line in path.read_text().splitlines()
            if line.startswith(('ATOM  ', 'HETATM'))]

protein_atoms = pdb_atom_lines(prepared_protein)
lipid_atoms = [line for line in pdb_atom_lines(inserted_pdb)
               if line.split()[3] == 'POPC']
assert protein_atoms and lipid_atoms, 'Missing protein or POPC atoms.'
assert all(line.split()[3] != 'POPC' for line in protein_atoms), (
    'Prepared protein unexpectedly contains POPC atoms.'
)

# Count unique lipid residues by the original PDB residue identifier.
lipid_residues = {(line[21], line[22:27]) for line in lipid_atoms}
print('Prepared protein atoms:', len(protein_atoms))
print('POPC atoms:', len(lipid_atoms))
print('POPC residues:', len(lipid_residues))
assert len(lipid_residues) == 200, (
    'Expected 200 POPC residues in the reference build. '
    'Inspect your actual build before proceeding.'
)

# Renumber serials and keep protein/lipid chain IDs distinct.
# Protein first, then POPC: PSF generation will use this exact PDB order.
combined_atoms = []
for serial, (line, chain_id) in enumerate(
    [(line, 'A') for line in protein_atoms] +
    [(line, 'L') for line in lipid_atoms], start=1
):
    assert serial <= 99999, 'PDB atom serial limit exceeded.'
    combined_atoms.append(f'{line[:6]}{serial:5d}{line[11:21]}{chain_id}{line[22:]}')

combined_pdb = output_dir_t3 / 'bacteriorhodopsin_POPC_loop_prepared.pdb'
combined_pdb.write_text('\n'.join(combined_atoms) + '\nEND\n')
print('Combined PDB:', combined_pdb)
print('Total coordinate atoms:', len(combined_atoms))

### 7.3 Generate the PSF topology

Use `namd-setup-psf` on the combined coordinates. This step should identify protein and POPC residues and create a topology compatible with the parameter files. The PSF and PDB **must describe the same atoms in the same order**.

Inspect the command output carefully: missing residues, unknown atom names, missing topology definitions or bond errors must be resolved before solvation or simulation.

In [ ]:
combined_psf = output_dir_t3 / 'bacteriorhodopsin_POPC_loop_prepared.psf'
!namd-setup-psf -i {combined_pdb} -o {combined_psf}

In [ ]:

from collections import Counter

def read_psf(path):
    """Read atom and bond records from a standard CHARMM/NAMD PSF."""
    lines = path.read_text().splitlines()

    def section(marker):
        for index, line in enumerate(lines):
            if marker in line:
                return index, int(line.split()[0])
        raise ValueError(f'Missing PSF section: {marker}')

    atom_start, n_atoms = section('!NATOM')
    atoms = []
    for line in lines[atom_start + 1:atom_start + 1 + n_atoms]:
        fields = line.split()
        assert len(fields) >= 8, f'Invalid PSF atom record: {line}'
        atoms.append({
            'id': int(fields[0]),
            'segid': fields[1],
            'resid': fields[2],
            'resname': fields[3],
            'name': fields[4],
            'charge': float(fields[6]),
        })

    bond_start, n_bonds = section('!NBOND')
    values = []
    for line in lines[bond_start + 1:]:
        if '!' in line:
            break
        values.extend(int(value) for value in line.split())
        if len(values) >= 2 * n_bonds:
            break

    assert len(values) == 2 * n_bonds, 'Incomplete PSF bond section.'
    bonds = {
        frozenset(values[i:i + 2])
        for i in range(0, len(values), 2)
    }
    return atoms, bonds


def validate_pdb_psf(pdb_lines, psf_path):
    """Verify PDB/PSF atom counts and atom-name ordering."""
    atoms, bonds = read_psf(psf_path)
    pdb_names = [line[12:16].strip() for line in pdb_lines]
    psf_names = [atom['name'] for atom in atoms]

    assert len(pdb_names) == len(psf_names), (
        f'PDB/PSF atom-count mismatch: {len(pdb_names)} / {len(psf_names)}'
    )
    mismatches = [
        (i, pdb_name, psf_name)
        for i, (pdb_name, psf_name) in
        enumerate(zip(pdb_names, psf_names), start=1)
        if pdb_name != psf_name
    ]
    assert not mismatches, f'PDB/PSF atom-name mismatches: {mismatches[:5]}'
    return atoms, bonds


def count_residues(atoms):
    """Count distinct residues using PSF segment and residue IDs."""
    residues = {
        (atom['segid'], atom['resid'], atom['resname'])
        for atom in atoms
    }
    return Counter(resname for _, _, resname in residues)


### 7.4 Validate the combined PDB/PSF

A successful command exit is not sufficient. Check that the atom counts agree and that the protein backbone is continuous. We use a lightweight standard-library PSF reader to inspect the generated topology and compare it with the PDB; this does not replace force-field validation or a short NAMD startup test.

In our previously prepared reference system, the protein contained **227 residues** and the final topology had **226 of 226 consecutive peptide bonds**. The checks below report the values for your newly generated system rather than assuming they match.

In [ ]:
assert combined_psf.is_file(), f'Missing PSF: {combined_psf}'
psf_atoms, psf_bonds = validate_pdb_psf(combined_atoms, combined_psf)
residue_counts = count_residues(psf_atoms)
protein_atoms_psf = [a for a in psf_atoms if a['resname'] in protein_resnames]
protein_keys = list(dict.fromkeys((a['segid'], a['resid']) for a in protein_atoms_psf))
print('PDB/PSF atoms:', len(combined_atoms), '/', len(psf_atoms))
print('POPC residues:', residue_counts['POPC'])
print('Protein atoms:', len(protein_atoms_psf), '| residues:', len(protein_keys))
assert residue_counts['POPC'] == 200, 'Unexpected POPC residue count.'
assert len(protein_keys) == 227, 'Unexpected protein residue count.'
backbone = {}
for atom in protein_atoms_psf:
    key = (atom['segid'], atom['resid'])
    if atom['name'] in ('C', 'N'):
        assert atom['name'] not in backbone.setdefault(key, {}), f'Duplicate backbone atom in {key}'
        backbone[key][atom['name']] = atom['id']
connected = sum(
    frozenset((backbone[left]['C'], backbone[right]['N'])) in psf_bonds
    for left, right in zip(protein_keys, protein_keys[1:])
    if 'C' in backbone.get(left, {}) and 'N' in backbone.get(right, {})
)
print(f'Consecutive protein C–N bonds: {connected}/{len(protein_keys)-1}')
assert connected == 226, 'Protein peptide connectivity differs from the reference.'
print('Combined PDB/PSF validation passed.')

### 7.5 Inspect the prepared system

Visualise the prepared PDB/PSF pair. Check that the protein is embedded within the membrane, that both leaflets are present and that the reconstructed loop is not obviously buried inside a lipid. A full assessment of close contacts and local geometry will be part of minimisation and equilibration.

**Next — Section 8:** Solvate the protein–lipid system, add ions, regenerate the solvated PSF, and check composition before generating NAMD configurations.

In [ ]:
import nglview as nv
prepared_view = nv.show_file(str(combined_pdb))
prepared_view.clear_representations()
prepared_view.add_cartoon(selection='protein', color='spectrum')
prepared_view.add_licorice(selection='POPC', opacity=0.45)
prepared_view.center()
prepared_view

## 8. Solvate and ionise the protein–POPC system

Our protein–lipid PDB/PSF pair describes a membrane in vacuum. Before running MD, we need an aqueous environment on both sides of the bilayer and ions to neutralise the system and approximate a chosen salt concentration.

We use the same `namd-solvate` tool as in Tutorial 2, but now the input includes **both POPC and bacteriorhodopsin**. For this example:

- `--layer_type bilayer` builds water layers around a membrane system.
- `--zlayer 20.0` requests 20 Å of water padding along the membrane normal (check the actual resulting box and water distribution).
- `--cation NA --anion CL` selects sodium and chloride ions.
- `--saltconc 0.15` requests a nominal **0.15 mol/L NaCl** concentration, in addition to any ions needed for charge neutralisation.

The ion number is discrete, so the achieved concentration is approximate. **Solvation changes the atom list:** regenerate the PSF from the solvated PDB rather than reusing the unsolvated PSF.

**Validation status:** The development workflow produced a neutral 64,448-atom system. The new notebook cells below have not yet been executed end-to-end; treat the expected counts as reference values, not universal requirements.

### 8.1 Add water and ions

The `--inp` argument takes the matching unsolvated PDB and PSF separated by a comma. Run this after Section 7 has generated and validated both files. Depending on your computer, solvation may take several minutes.

The output is written to the Tutorial 3 output directory. Keep the source PDB/PSF unchanged so you can reproduce this step.

In [ ]:
solvated_pdb = output_dir_t3 / 'bacteriorhodopsin_POPC_solvated.pdb'
assert combined_pdb.is_file() and combined_psf.is_file(), 'Complete Section 7 first.'
print('Input PDB:', combined_pdb)
print('Input PSF:', combined_psf)
print('Solvated PDB:', solvated_pdb)

In [ ]:
!namd-solvate --inp {combined_pdb},{combined_psf} --layer_type bilayer --out {solvated_pdb} --zlayer 20.0 --cation NA --anion CL --saltconc 0.15

### 8.2 Regenerate the solvated PSF

Water and ions must be represented in the topology, with the same atom ordering as the new coordinates. Use `namd-setup-psf` on the **solvated** PDB. Review the output for unknown residues, topology errors and malformed PDB records before continuing.

In the development run, one PDB-format issue arose when residue numbers exceeded the four-column PDB limit; do not silently ignore parsing warnings.

In [ ]:
solvated_psf = output_dir_t3 / 'bacteriorhodopsin_POPC_solvated.psf'
assert solvated_pdb.is_file(), f'Missing solvated PDB: {solvated_pdb}'
!namd-setup-psf -i {solvated_pdb} -o {solvated_psf}

### 8.3 Check the final system composition and net charge

We verify that the PDB and PSF have matching atom counts **and atom names in the same order**, and count the major molecular components. The final PSF also supplies per-atom charges, so we can calculate the net system charge.

Our reference build contained **200 POPC, 227 protein residues, 11,337 TIP3 waters, 31 SOD, 31 CLA and 64,448 atoms**. A fresh solvation can yield different water/ion counts. A neutral system should have total charge close to zero (allowing floating-point tolerance).

This structural check is necessary but not sufficient: the force-field parameter files must also load successfully in NAMD, and the minimised structure must be assessed before equilibration.

In [ ]:
assert solvated_pdb.is_file() and solvated_psf.is_file(), 'Solvation or PSF generation did not complete.'
solvated_pdb_atoms = [line for line in solvated_pdb.read_text().splitlines()
                     if line.startswith(('ATOM  ', 'HETATM'))]
solvated_atoms, solvated_bonds = validate_pdb_psf(solvated_pdb_atoms, solvated_psf)
residue_counts = count_residues(solvated_atoms)
protein_residue_keys = {(a['segid'], a['resid']) for a in solvated_atoms
                        if a['resname'] in protein_resnames}
net_charge = sum(a['charge'] for a in solvated_atoms)
print('Total atoms:', len(solvated_atoms))
print('POPC lipids:', residue_counts['POPC'])
print('TIP3 waters:', residue_counts['TIP3'])
print('SOD ions:', residue_counts['SOD'])
print('CLA ions:', residue_counts['CLA'])
print('Protein residues:', len(protein_residue_keys))
print('Net PSF charge (e):', round(net_charge, 6))
assert residue_counts['POPC'] == 200, 'Unexpected POPC count.'
assert len(protein_residue_keys) == 227, 'Unexpected protein residue count.'
assert residue_counts['TIP3'] > 0, 'No TIP3 waters found.'
assert abs(net_charge) < 1e-3, 'System is not electrically neutral.'
print('Solvated PDB/PSF validation passed.')

### 8.4 Inspect hydration and ion placement

Display the protein as a cartoon, POPC as sticks and ions as spheres. Water is initially hidden to keep the view readable; enable it when checking whether both sides of the membrane are hydrated. Check for ions inside the hydrophobic membrane core, water gaps and unusual overlaps.

**Discussion:** Why do we regenerate the PSF after solvation? What is the difference between charge neutralisation and specifying a bulk salt concentration?

**Next — Section 9:** Generate NAMD minimisation, two protein-aware equilibration stages and production configuration files. The reference equilibration and short production runs completed successfully. Longer simulations and additional structural checks are required before interpreting equilibrium properties.

In [ ]:
import nglview as nv
solvated_view = nv.show_file(str(solvated_pdb))
solvated_view.clear_representations()
solvated_view.add_cartoon(selection='protein', color='spectrum')
solvated_view.add_licorice(selection='POPC', opacity=0.5)
solvated_view.add_spacefill(selection='SOD or CLA', radius_scale=0.6)
# Uncomment to show water (can be slow for a large system):
# solvated_view.add_line(selection='TIP3', opacity=0.15)
solvated_view.center()
solvated_view

## 9. Generate NAMD minimisation and equilibration configurations

Our solvated protein–membrane system is now ready for the next preparation step: generating the NAMD configuration files.

Unlike a lipid-only bilayer, this system contains a membrane protein with a reconstructed loop. We therefore use a **staged equilibration protocol**, progressively releasing positional restraints.

Shapespyer provides this through the existing `namd-setup-equil` command with the optional `--protein-equil` flag.

### 9.1 The four-stage protocol

| Stage | Protein backbone | POPC phosphorus | Purpose |
|---|---|---|---|
| Minimisation | Restrained | Restrained | Relax unfavourable atomic contacts |
| Equilibration 1 | Restrained | Restrained | Relax solvent, side chains and surrounding environment |
| Equilibration 2 | Unrestrained | Restrained | Allow protein backbone relaxation |
| Production | Unrestrained | Unrestrained | Sample dynamics without positional restraints |

The positional restraints are harmonic, with `consexp 2` in the NAMD configuration. Their atom-specific coefficients are supplied through the generated `.fix` files.

The default lipid-only Shapespyer workflow remains a three-stage protocol; the additional equilibration stage is enabled only with `--protein-equil`.

**Important:** Successful minimisation and equilibration do not automatically establish physical equilibrium. Structural properties, energies and membrane dimensions must be assessed before interpreting production trajectories.

For this workshop, we will demonstrate configuration generation and a short minimisation. Longer simulations will be illustrated using separately prepared reference trajectories.


### 9.2 Generate the four NAMD configuration files

We use `namd-setup-equil --protein-equil` to generate minimisation, two equilibration stages and production configurations.

The input structure is the solvated PDB/PSF pair created in Section 8. The `--struct` argument takes their common filename prefix, without the `.pdb` or `.psf` extension.

We save the configurations in a separate `configs/` directory. Shapespyer generates paths relative to each configuration file, so the PSF, PDB, CHARMM36 parameter files and restraint files can remain in the parent directory.

For the workshop, we request a short minimisation and retain longer equilibration and production stages for reference simulations.

**Note:** Generating these files does not execute NAMD.


In [ ]:
import subprocess
from pathlib import Path

config_dir_t3 = output_dir_t3 / 'configs'
config_dir_t3.mkdir(parents=True, exist_ok=True)

structure_root = solvated_pdb.with_suffix('')

namd_command = [
    'namd-setup-equil',
    '--struct', str(structure_root),
    '--out_min', str(config_dir_t3 / 'min.conf'),
    '--out_equil', str(config_dir_t3 / 'equil.conf'),
    '--out_equil2', str(config_dir_t3 / 'equil2.conf'),
    '--out_prod', str(config_dir_t3 / 'prod_1.conf'),
    '--protein-equil',
    '--temp', '300',
    '--steps_min', '10000',
    '--steps_equil', '100000',
    '--steps_equil2', '100000',
    '--steps_prod', '100000',
]

print('Generating NAMD configurations...')
subprocess.run(namd_command, check=True)

for filename in ('min.conf', 'equil.conf', 'equil2.conf', 'prod_1.conf'):
    path = config_dir_t3 / filename
    assert path.is_file(), f'Missing configuration: {path}'
    print('Created:', path)


### 9.3 Validate the generated restraint files

Before running NAMD, we verify that the positional restraints target the intended atoms.

The combined restraint file should contain the protein backbone atoms (`N`, `CA`, `C`, `O`) and one phosphorus atom per POPC molecule. The lipid-only file should contain only the POPC phosphorus atoms.

NAMD reads the restraint coefficients from the PDB B-factor column. Here, `1.00` indicates a restrained atom and `0.00` an unrestrained atom.

The expected counts for our reference system are:

| Restraint file | Protein backbone | POPC phosphorus | Total |
|---|---:|---:|---:|
| Combined | 909 | 200 | 1107 |
| Lipid-only | 0 | 200 | 200 |

These counts are specific to this reference structure; other systems may have different numbers of protein and lipid atoms.


In [ ]:
from collections import Counter

def count_restraints(path):
    counts = Counter()
    with path.open() as handle:
        for line in handle:
            if not line.startswith(('ATOM  ', 'HETATM')):
                continue
            if float(line[60:66]) <= 0:
                continue

            atom = line[12:16].strip()
            residue = line[17:21].strip()
            segment = line[72:76].strip()

            if segment == 'PROT' and atom in {'N', 'CA', 'C', 'O'}:
                counts['protein_backbone'] += 1
            elif residue == 'POPC' and atom == 'P':
                counts['lipid_phosphorus'] += 1
            else:
                counts['unexpected'] += 1

    return counts

combined_fix = Path(str(structure_root) + '_lipid_protein.fix')
lipid_fix = Path(str(structure_root) + '.fix')

combined_counts = count_restraints(combined_fix)
lipid_counts = count_restraints(lipid_fix)

print('Combined restraints:', dict(combined_counts))
print('Lipid-only restraints:', dict(lipid_counts))

assert combined_counts['protein_backbone'] > 0
assert combined_counts['lipid_phosphorus'] == 200
assert combined_counts['unexpected'] == 0

assert lipid_counts['protein_backbone'] == 0
assert lipid_counts['lipid_phosphorus'] == 200
assert lipid_counts['unexpected'] == 0

print('Restraint validation passed.')


### 9.4 Optional: Run a short NAMD minimisation

As in Tutorial 2, running NAMD is optional. The shared workshop
notebook provides `RUN_NAMD` and `namd_exe`.

For this exercise, we create a separate **100-step minimisation**
configuration from `min.conf`. This preserves the original
10,000-step configuration for longer reference simulations.

The demonstration uses the same protein-backbone and POPC-phosphorus
restraints validated in Section 9.3.

**Important:** 100 minimisation steps demonstrate the workflow;
they do not produce an equilibrated membrane–protein system.

The demonstration runs only when `RUN_NAMD = True`. Otherwise,
participants can continue to the reference trajectory analysis.


In [ ]:
import re
import subprocess

# Create a demonstration configuration without changing min.conf.
full_min_config = config_dir_t3 / 'min.conf'
demo_min_config = config_dir_t3 / 'min_demo.conf'

config_text = full_min_config.read_text()

# Replace the minimisation step count, preserving other settings.
config_text, replacements = re.subn(
    r'(?m)^(\s*minimize\s+)\d+',
    r'\g<1>100',
    config_text,
)
assert replacements == 1, (
    'Expected exactly one minimize directive in min.conf.'
)

# Use a separate output prefix to protect reference results.
config_text, output_replacements = re.subn(
    r'(?m)^(\s*outputName\s+)\S+',
    r'\g<1>min_demo',
    config_text,
)
assert output_replacements == 1, (
    'Expected exactly one outputName directive.'
)

demo_min_config.write_text(config_text)
print('Created demonstration configuration:', demo_min_config)

if globals().get('RUN_NAMD', False):
    assert namd_exe is not None and Path(namd_exe).is_file(), (
        'NAMD is not installed. Run the shared workshop setup first.'
    )

    log_file = config_dir_t3 / 'min_demo.log'
    with log_file.open('w') as log:
        result = subprocess.run(
            [str(namd_exe), '+p2', demo_min_config.name],
            cwd=config_dir_t3,
            stdout=log,
            stderr=subprocess.STDOUT,
            text=True,
            timeout=600,
        )

    print('NAMD return code:', result.returncode)
    print('\\nLast 20 lines of the log:')
    print('\\n'.join(
        log_file.read_text(errors='replace').splitlines()[-20:]
    ))
    assert result.returncode == 0, (
        f'NAMD minimisation failed. Inspect {log_file}'
    )
else:
    print('NAMD minimisation skipped (optional exercise).')


### 9.5 Understanding the NAMD restart sequence

The four configuration files form a sequential MD workflow:

| Configuration | Input | Output |
|---|---|---|
| `min.conf` | Initial solvated PDB/PSF | `*_min` |
| `equil.conf` | Minimisation restart | `*_equil` |
| `equil2.conf` | Equilibration 1 restart | `*_equil2` |
| `prod_1.conf` | Equilibration 2 restart | `*_prod` |

Each stage reads the preceding stage's restart files:

- `.coor` — atomic coordinates
- `.vel` — atomic velocities
- `.xsc` — periodic simulation cell

The configurations must be executed **in order**, from the `configs/`
directory, so that their relative file paths resolve correctly.

For this example, the two equilibration stages and production each
request 100,000 steps at a 1 fs timestep, corresponding to **100 ps
per stage**.

These durations are illustrative, not sufficient to establish
equilibrium or obtain converged production statistics.

**Workshop execution:** We run only the optional 100-step minimisation
demonstration. The complete staged workflow is shown for explanation,
and longer simulation results will be provided separately.

**Important:** `min_demo.conf` produces independent demonstration
output. It does not replace the full `min.conf` stage in the restart
sequence.

The generated configurations use `firsttimestep 0` for each stage.
When combining trajectories, construct a continuous time axis from
their actual step counts and timesteps rather than treating all
frames as belonging to one uninterrupted simulation clock.



## 10. Inspecting the simulation results

The preparation and NAMD configuration stages are now complete.
We can use reference simulation outputs to examine how the
protein–membrane system responds during equilibration.

For the workshop, these analyses use **precomputed reference data**,
so participants do not need to wait for long NAMD simulations.

We will examine:

1. Changes in the periodic simulation cell and projected membrane area.
2. Protein structural stability during equilibration.
3. The reconstructed protein loop and its membrane environment.
4. Differences between restrained equilibration and unrestrained production.

These analyses help identify potential problems, but a short,
successfully completed simulation does not establish thermodynamic
equilibrium or convergence.

### 10.1. Monitoring membrane dimensions

NAMD writes periodic-cell information to `.xst` files.
Each record contains the simulation timestep and the three
periodic-cell vectors.

The projected membrane area is calculated as

\[
A_{xy} = |\mathbf{a} \times \mathbf{b}|
\]

where `a` and `b` are the two in-plane cell vectors.

For a membrane containing a protein, this is the **total
simulation-cell area**, not the area per lipid.


In [ ]:

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

reference_dir = Path("outputs/tutorial_03/reference")

xst_files = {
    "Equilibration 1": reference_dir / "bacteriorhodopsin_POPC_solvated_equil_100ps.xst",
    "Equilibration 2": reference_dir / "bacteriorhodopsin_POPC_solvated_equil2_100ps.xst",
    "Production": reference_dir / "bacteriorhodopsin_POPC_solvated_prod_100ps.xst",
}

def read_xst_area(path, timestep_fs=1.0):
    data = np.loadtxt(path, comments="#", ndmin=2)

    steps = data[:, 0]
    a = data[:, 1:4]
    b = data[:, 4:7]

    area = np.linalg.norm(np.cross(a, b), axis=1)
    time_ps = (steps - steps[0]) * timestep_fs / 1000.0

    return time_ps, area

fig, ax = plt.subplots(figsize=(8, 4.5))

for label, path in xst_files.items():
    if not path.is_file():
        print(f"Reference file not available: {path}")
        continue

    time_ps, area = read_xst_area(path)
    ax.plot(time_ps, area, label=label)

ax.set_xlabel("Time within stage (ps)")
ax.set_ylabel("Projected simulation-cell area (Å²)")
ax.set_title("Membrane–protein system: equilibration and production")
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()



#### Interpreting the membrane-area evolution

The reference simulations show two different behaviours:

- **Equilibration 1:** The projected area increases by approximately
  3.6%, indicating that the initially constructed membrane is
  undergoing substantial relaxation.
- **Equilibration 2:** After releasing the protein backbone restraints,
  the projected area fluctuates around a relatively stable value,
  without the pronounced upward trend observed during the first stage.

This suggests that the membrane is becoming more stable under the
chosen simulation conditions.

However, **100 ps is insufficient to demonstrate membrane
equilibration**. Longer simulations and additional observables,
including protein stability, lipid organisation and membrane thickness,
would be needed before drawing conclusions about equilibrium
properties.

**Discussion question:** Why is the total projected simulation-cell
area not equivalent to the area per lipid in a membrane containing
a protein?



### 10.2. Protein and reconstructed-loop RMSD

After the restrained equilibration stages, we ran a short **100 ps
unrestrained NPT production simulation**.

The reference RMSD values were calculated in VMD from 100 trajectory
frames, aligning each frame to the first production frame using the
protein backbone.

We compare:

- **Protein backbone RMSD:** overall structural displacement.
- **Reconstructed loop RMSD:** displacement of residues 157–161
  after the same backbone alignment.

This is a short validation example, not a converged production analysis.


In [ ]:

import numpy as np
import matplotlib.pyplot as plt

rmsd_file = reference_dir / "bacteriorhodopsin_POPC_prod_100ps_rmsd.dat"

if rmsd_file.is_file():
    rmsd = np.loadtxt(rmsd_file, comments="#", ndmin=2)

    assert rmsd.shape[1] == 3, "Expected frame, backbone RMSD, loop RMSD"

    frames = rmsd[:, 0]
    backbone_rmsd = rmsd[:, 1]
    loop_rmsd = rmsd[:, 2]

    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.plot(frames, backbone_rmsd, label="Protein backbone")
    ax.plot(frames, loop_rmsd, label="Reconstructed loop (157–161)")

    ax.set_xlabel("Production trajectory frame")
    ax.set_ylabel("RMSD from first production frame (Å)")
    ax.set_title("Protein RMSD during 100 ps production")
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

    print(f"Backbone mean RMSD: {backbone_rmsd.mean():.3f} Å")
    print(f"Loop mean RMSD: {loop_rmsd.mean():.3f} Å")
else:
    print(f"Reference RMSD file not available: {rmsd_file}")



#### Interpreting the RMSD

In the 100 ps reference production simulation:

- The mean protein-backbone RMSD was approximately **1.05 Å**.
- The mean reconstructed-loop RMSD was approximately **2.14 Å**.

The loop is more mobile than the overall backbone, which is
consistent with its reconstructed geometry and local flexibility.

These results demonstrate that the prepared system completed a short
unrestrained simulation without an obvious large backbone displacement.
They do **not** validate the reconstructed loop geometry or establish
long-term structural stability.

**Workshop takeaway:** Shapespyer prepares the molecular system;
subsequent minimisation, equilibration and validation remain essential
before interpreting physical properties.
